# LeetCode 450 — Delete Node in a BST bằng Python

Link đề: https://leetcode.com/problems/delete-node-in-a-bst/ · Độ khó: **Medium** · Chủ đề: Tree, Binary Search Tree, Binary Tree, Recursion

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def deleteNode(self, root: Optional[TreeNode], key: int) -> Optional[TreeNode]:
        ...
```

Ghi chú: đề được viết lại theo hiểu biết về đề gốc; nên đối chiếu lại đề gốc nếu muốn khớp từng chữ (đặc biệt phần ví dụ hình minh hoạ). Bài này có **nhiều đáp án đúng** (xoá nút hai con có thể thay bằng nút kế nhiệm hoặc nút tiền nhiệm), nên notebook **kiểm tra bằng hàm kiểm tra BST hợp lệ + đúng tập giá trị**, không so đáp án cứng. Notebook tự dựng `TreeNode`, `build_tree`, `tree_to_list`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho gốc của một BST và một giá trị `key`; **xoá nút có giá trị `key`** (nếu có) sao cho kết quả **vẫn là BST**, rồi trả về gốc (có thể đã đổi) của cây.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, đệ quy trả về **gốc của cây con** (`root.left = f(root.left, key)`), vòng `while` đi xuống con trái/phải, duyệt trung tự (inorder) để lấy dãy giá trị, và viết hàm **kiểm tra BST hợp lệ**.
3. Đi từ **vét: lấy dãy trung tự, bỏ `key`, dựng lại cây cân bằng** → **thay bằng nút kế nhiệm (copy giá trị)** → **treo cả cây con trái vào nút trái nhất của cây con phải** → **nối lại chính nút tiền nhiệm (không copy giá trị)** → **bản lặp dùng con trỏ cha (O(1) bộ nhớ phụ)**.
4. Hiểu mấu chốt: xoá nút có ba trường hợp theo số con (0, 1, 2 con); chỉ trường hợp **hai con** cần "người kế vị": nút nhỏ nhất ở cây con phải (kế nhiệm) hoặc nút lớn nhất ở cây con trái (tiền nhiệm) để giữ nguyên thứ tự trung tự.
5. Tránh các lỗi kinh điển: quên gán lại con (`root.left = ...`), làm mất cây con khi xoá nút hai con, thay nhầm bằng nút lớn nhất của cây **phải**, quên xoá nút kế nhiệm khỏi chỗ cũ (giá trị bị lặp), và quên xử lý `root is None`.

## 2. Đề bài & ví dụ

Cho **gốc** của một cây tìm kiếm nhị phân (BST) và một giá trị `key`. Hãy **xoá nút có giá trị `key`** khỏi BST và trả về **gốc (có thể đã được cập nhật)** của BST.

Việc xoá chia thành hai giai đoạn:

1. **Tìm** nút cần xoá.
2. Nếu thấy, **xoá** nút đó.

**Ví dụ 1**

- Input: `root = [5,3,6,2,4,null,7]`, `key = 3`
- Output: `[5,4,6,2,null,null,7]`
- Giải thích: tìm thấy nút `3` và xoá. Đáp án hợp lệ khác: `[5,2,6,null,4,null,7]` — **cũng được chấp nhận**.

**Ví dụ 2**

- Input: `root = [5,3,6,2,4,null,7]`, `key = 0`
- Output: `[5,3,6,2,4,null,7]`
- Giải thích: cây không có nút giá trị `0`, giữ nguyên.

**Ví dụ 3**

- Input: `root = []`, `key = 0`
- Output: `[]`

**Ràng buộc:**

- Số nút của cây trong khoảng `[0, 10^4]`.
- `-10^5 <= Node.val <= 10^5`
- Mỗi nút có giá trị **duy nhất**.
- `root` là một BST hợp lệ.
- `-10^5 <= key <= 10^5`

**Follow up:** giải với thời gian `O(chiều cao cây)`.

Chú ý: do **nhiều đáp án đúng**, ta kiểm tra bằng **tính chất**: (a) kết quả là BST hợp lệ; (b) tập giá trị = tập ban đầu bỏ `key`; (c) nếu `key` không có trong cây thì cây giữ nguyên.

In [1]:
from typing import Optional, List
from collections import deque
import random
import sys
import time


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


examples = [
    ([5, 3, 6, 2, 4, None, 7], 3),
    ([5, 3, 6, 2, 4, None, 7], 0),
    ([], 0),
]
for vals, key in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "key =", key)

[5, 3, 6, 2, 4, None, 7] key = 3
[5, 3, 6, 2, 4, None, 7] key = 0
[] key = 0


## 3. Kiến thức nền

### 3.1. BST và việc tìm một giá trị

Trong BST, mọi giá trị ở cây con trái `<` nút, mọi giá trị ở cây con phải `>` nút. Vì vậy tìm `key` là đi một đường từ gốc: `key < node.val` thì sang trái, `key > node.val` thì sang phải, bằng thì tìm thấy. Mỗi bước loại bỏ một nửa (nếu cây cân bằng): `O(h)`.

In [2]:
def find(root, key):
    node = root
    while node and node.val != key:
        node = node.left if key < node.val else node.right
    return node


t = build_tree([5, 3, 6, 2, 4, None, 7])
print(find(t, 4).val, find(t, 7).val, find(t, 0))      # 4 7 None

4 7 None


### 3.2. Duyệt trung tự (inorder) cho ra dãy tăng dần

Duyệt *trái → nút → phải* trên BST cho dãy giá trị **tăng dần**. Ta viết bản **vòng lặp dùng stack** (không đệ quy) để chạy được cả với cây sâu 10^4 nút.

In [3]:
def inorder(root):
    out, stack, cur = [], [], root
    while stack or cur:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        out.append(cur.val)
        cur = cur.right
    return out


print(inorder(build_tree([5, 3, 6, 2, 4, None, 7])))     # [2, 3, 4, 5, 6, 7]

[2, 3, 4, 5, 6, 7]


### 3.3. Nút kế nhiệm (successor) và tiền nhiệm (predecessor)

- **Kế nhiệm** của nút `x` có cây con phải: là nút **nhỏ nhất** của cây con phải — tức là nút **trái nhất** (đi `left` mãi đến khi hết).
- **Tiền nhiệm**: nút **lớn nhất** của cây con trái — đi `right` mãi.

Trong thứ tự trung tự, kế nhiệm đứng ngay **sau** `x`, tiền nhiệm đứng ngay **trước** `x`. Thay nút `x` bằng một trong hai nút này thì **thứ tự trung tự không bị phá** (chỉ thiếu `x`).

In [4]:
def leftmost(node):
    while node.left:
        node = node.left
    return node


def rightmost(node):
    while node.right:
        node = node.right
    return node


t = build_tree([5, 3, 6, 2, 4, None, 7])
print("kế nhiệm của 5   :", leftmost(t.right).val)       # 6
print("tiền nhiệm của 5 :", rightmost(t.left).val)       # 4
print("trung tự         :", inorder(t))                  # ... 4, 5, 6 ...

kế nhiệm của 5   : 6
tiền nhiệm của 5 : 4
trung tự         : [2, 3, 4, 5, 6, 7]


### 3.4. Mẫu đệ quy "trả về gốc của cây con"

Rất nhiều bài cây dạng "sửa cây" dùng mẫu: hàm `f(node)` **trả về gốc mới của cây con** sau khi sửa. Cha gán lại kết quả: `node.left = f(node.left)`. Nhờ đó "xoá nút" tự nhiên: hàm trả về **một nút khác (hoặc `None`)** thay cho nút bị xoá, và cha chỉ cần nối lại.

In [5]:
def remove_leaf_demo(node, key):
    """Ví dụ nhỏ: chỉ xoá nếu nút key là LÁ; trả về gốc mới của cây con."""
    if node is None:
        return None
    if key < node.val:
        node.left = remove_leaf_demo(node.left, key)    # gán lại kết quả cho cha
    elif key > node.val:
        node.right = remove_leaf_demo(node.right, key)
    elif node.left is None and node.right is None:
        return None                                      # xoá lá: cha sẽ nhận None
    return node


t = build_tree([5, 3, 6, 2, 4, None, 7])
t = remove_leaf_demo(t, 2)
print(tree_to_list(t))        # [5, 3, 6, None, 4, None, 7]

[5, 3, 6, None, 4, None, 7]


### 3.5. Hàm kiểm tra BST hợp lệ (để chấm bài nhiều đáp án)

Một cây là BST hợp lệ khi **mỗi nút nằm trong khoảng `(lo, hi)`** do tổ tiên quy định: gốc `(-inf, inf)`; con trái của `x` thuộc `(lo, x)`; con phải thuộc `(x, hi)`. Ngoài ra ta kiểm tra **không có nút dùng chung** (cùng một nút bị nối hai chỗ) — lỗi hay gặp khi nối con trỏ sai.

In [6]:
def is_valid_bst(root):
    inf = float("inf")
    seen = set()
    stack = [(root, -inf, inf)]
    while stack:
        node, lo, hi = stack.pop()
        if node is None:
            continue
        if id(node) in seen:              # một nút xuất hiện hai lần -> cấu trúc hỏng
            return False
        seen.add(id(node))
        if not (lo < node.val < hi):
            return False
        stack.append((node.left, lo, node.val))
        stack.append((node.right, node.val, hi))
    return True


good = build_tree([5, 3, 6, 2, 4, None, 7])
bad = build_tree([5, 3, 6, 2, 7])            # 7 nằm ở cây con trái của 3 nhưng lớn hơn 5
print(is_valid_bst(good), is_valid_bst(bad), is_valid_bst(None))   # True False True

True False True


### 3.6. Bộ kiểm tra tổng quát cho bài nhiều đáp án

`check_delete(fn, vals, key)` dựng một cây **mới** từ `vals` (vì hàm xoá **sửa trực tiếp** cây), gọi `fn`, rồi kiểm tra ba tính chất. Tham số `shape_if_absent=False` dành cho cách (như dựng lại cây) không bảo đảm giữ nguyên hình dạng khi không có `key`.

In [7]:
def check_delete(fn, vals, key, shape_if_absent=True):
    root = build_tree(vals)
    before = inorder(root)
    out = fn(root, key)
    assert is_valid_bst(out), ("không còn là BST", vals, key, tree_to_list(out))
    expected = [v for v in before if v != key]
    assert inorder(out) == expected, ("sai tập giá trị", vals, key, tree_to_list(out))
    if shape_if_absent and key not in before:
        assert tree_to_list(out) == tree_to_list(build_tree(vals)), ("đổi cây khi không có key", vals, key)
    return out


# thử với một "hàm xoá" giả: không làm gì (đúng nếu key không có, sai nếu có)
identity = lambda root, key: root
check_delete(identity, [5, 3, 6], 100)
try:
    check_delete(identity, [5, 3, 6], 3)
except AssertionError as e:
    print("Bắt được lỗi như mong đợi:", e.args[0][0])

Bắt được lỗi như mong đợi: sai tập giá trị


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét BST `[5,3,6,2,4,null,7]`:

```
        5
       / \
      3   6
     / \   \
    2   4   7
```

Sau khi **tìm** thấy nút cần xoá, tuỳ **số con** của nó:

1. **Xoá lá** (ví dụ `2`): bỏ luôn — cha trỏ `None`.
2. **Xoá nút một con** (ví dụ `6`, chỉ có con phải `7`): **nhấc** con lên thay nút: `5.right = 7`.
3. **Xoá nút hai con** (ví dụ `3`, con `2` và `4`): không thể nhấc cả hai con lên. Phải chọn **một nút khác** thay chỗ `3` mà vẫn giữ thứ tự:
   - Dãy trung tự là `2 3 4 5 6 7`. Bỏ `3` thì còn `2 4 5 6 7`.
   - Nút đứng ngay **sau** `3` là `4` (kế nhiệm), ngay **trước** là `2` (tiền nhiệm). Đưa một trong hai vào vị trí của `3`:
     - Dùng kế nhiệm `4`: cây `[5,4,6,2,null,null,7]`.
     - Dùng tiền nhiệm `2`: cây `[5,2,6,null,4,null,7]`.

Cả hai đều là đáp án hợp lệ (ví dụ 1 của đề nói rõ). Kiểm chứng bằng code rằng cả hai kết quả đều hợp lệ và có đúng tập giá trị:

In [8]:
a = build_tree([5, 4, 6, 2, None, None, 7])
b = build_tree([5, 2, 6, None, 4, None, 7])
for name, t in [("dùng kế nhiệm", a), ("dùng tiền nhiệm", b)]:
    print(name, tree_to_list(t), "hợp lệ:", is_valid_bst(t), "trung tự:", inorder(t))
    assert is_valid_bst(t) and inorder(t) == [2, 4, 5, 6, 7]

dùng kế nhiệm [5, 4, 6, 2, None, None, 7] hợp lệ: True trung tự: [2, 4, 5, 6, 7]
dùng tiền nhiệm [5, 2, 6, None, 4, None, 7] hợp lệ: True trung tự: [2, 4, 5, 6, 7]


## 5. Cách 1 — Brute Force / baseline: lấy dãy trung tự, bỏ `key`, **dựng lại cây**

**Ý tưởng.** Dùng thẳng khái niệm "BST = dãy đã sắp xếp":

1. Lấy dãy trung tự `vals` (tăng dần).
2. Bỏ `key` (nếu có).
3. Dựng lại một BST **cân bằng** từ dãy đã sắp xếp: lấy phần tử giữa làm gốc, đệ quy cho nửa trái và nửa phải.

**Nút thắt.** Luôn tốn `O(n)` thời gian và bộ nhớ — kể cả khi chỉ xoá một lá — trong khi đề hỏi `O(h)`. Ngoài ra nó **phá hình dạng** cây (kể cả khi `key` không tồn tại, cây vẫn bị "xếp lại"), tạo ra rất nhiều nút mới. Nhưng nó **đúng** nên rất hợp làm **oracle** để đối chiếu tập giá trị.

In [9]:
def delete_rebuild(root, key):
    vals = [v for v in inorder(root) if v != key]

    def build(l, r):
        if l > r:
            return None
        m = (l + r) // 2
        return TreeNode(vals[m], build(l, m - 1), build(m + 1, r))

    return build(0, len(vals) - 1)


for vals, key in examples:
    out = check_delete(delete_rebuild, vals, key, shape_if_absent=False)
    print(vals, "xoá", key, "->", tree_to_list(out))

[5, 3, 6, 2, 4, None, 7] xoá 3 -> [5, 2, 6, None, 4, None, 7]
[5, 3, 6, 2, 4, None, 7] xoá 0 -> [4, 2, 6, None, 3, 5, 7]
[] xoá 0 -> []


**Độ phức tạp.** Thời gian `O(n)`, bộ nhớ `O(n)` (danh sách + cây mới). Đúng, nhưng quá tay cho một thao tác cục bộ.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** chỉ **nút bị xoá** và **một nút thay thế** bị ảnh hưởng. Ta chỉ cần:

- **Tìm** nút bằng cách đi một đường từ gốc (như tìm kiếm BST) — `O(h)`.
- **Nối lại** tại chỗ: nút bị xoá được thay bằng:
  - `None` (lá);
  - con duy nhất (một con);
  - kế nhiệm hoặc tiền nhiệm (hai con), hoặc dùng mẹo "treo cả cây con trái vào nút trái nhất của cây con phải".

Mọi nút khác giữ nguyên → `O(h)` thời gian. Ba trường hợp có thể phân loại chỉ bằng hai câu hỏi: `node.left is None?`, `node.right is None?`.

In [10]:
def classify(node):
    if node.left is None and node.right is None:
        return "lá (0 con)"
    if node.left is None or node.right is None:
        return "một con"
    return "hai con"


t = build_tree([5, 3, 6, 2, 4, None, 7])
for key in [2, 6, 3, 5]:
    print(f"nút {key}: {classify(find(t, key))}")

nút 2: lá (0 con)
nút 6: một con
nút 3: hai con
nút 5: hai con


## 7. Các cách tiếp theo

### Cách 2 — Thay bằng **kế nhiệm** (sao chép giá trị rồi xoá kế nhiệm ở cây phải)

**Giải thích từng đoạn.**

1. `root is None` → trả `None` (không tìm thấy).
2. `key < root.val`: `root.left = f(root.left, key)`; `key > root.val`: `root.right = f(root.right, key)`; trả về `root`.
3. Tìm thấy (`key == root.val`):
   - Không có con trái → trả `root.right` (bao gồm cả trường hợp lá).
   - Không có con phải → trả `root.left`.
   - Hai con: tìm kế nhiệm `succ` (trái nhất của `root.right`), **chép giá trị** `root.val = succ.val`, rồi **xoá `succ.val` khỏi cây phải** bằng đệ quy (nó có tối đa một con nên rơi vào trường hợp dễ).

In [11]:
def delete_copy_successor(root, key):
    if root is None:
        return None
    if key < root.val:
        root.left = delete_copy_successor(root.left, key)
        return root
    if key > root.val:
        root.right = delete_copy_successor(root.right, key)
        return root
    if root.left is None:
        return root.right
    if root.right is None:
        return root.left
    succ = root.right
    while succ.left:
        succ = succ.left
    root.val = succ.val                                    # sao chép giá trị kế nhiệm
    root.right = delete_copy_successor(root.right, succ.val)   # xoá kế nhiệm ở chỗ cũ
    return root


for vals, key in examples:
    out = check_delete(delete_copy_successor, vals, key)
    print(vals, "xoá", key, "->", tree_to_list(out))

[5, 3, 6, 2, 4, None, 7] xoá 3 -> [5, 4, 6, 2, None, None, 7]
[5, 3, 6, 2, 4, None, 7] xoá 0 -> [5, 3, 6, 2, 4, None, 7]
[] xoá 0 -> []


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def deleteNode(self, root: Optional[TreeNode], key: int) -> Optional[TreeNode]:
        if root is None:
            return None
        if key < root.val:
            root.left = self.deleteNode(root.left, key)
            return root
        if key > root.val:
            root.right = self.deleteNode(root.right, key)
            return root
        if root.left is None:
            return root.right
        if root.right is None:
            return root.left
        succ = root.right
        while succ.left:
            succ = succ.left
        root.val = succ.val
        root.right = self.deleteNode(root.right, succ.val)
        return root
```

**Độ phức tạp.** Thời gian `O(h)` (một đường tìm nút + một đường tìm/ xoá kế nhiệm, tổng ≤ `2h`); bộ nhớ `O(h)` ngăn xếp đệ quy.

### Cách 3 — **Treo cả cây con trái** vào nút trái nhất của cây con phải

**Ý tưởng khác.** Khi xoá nút `x` có hai con: mọi giá trị của cây trái `L` đều **nhỏ hơn** mọi giá trị của cây phải `R`. Nút nhỏ nhất của `R` (trái nhất, gọi `m`) **chưa có con trái**, nên ta có thể **treo nguyên `L`** làm con trái của `m`, rồi dùng `R` thay cho `x`. Không sao chép giá trị, không xoá thêm nút nào, chỉ **một lần đi xuống** để tìm `m`.

**Giải thích từng đoạn.** Giống Cách 2 ở phần tìm kiếm; khác ở nhánh hai con:

```python
node = root.right
while node.left: node = node.left     # node = nút trái nhất của cây phải
node.left = root.left                 # treo cả cây trái vào đó
return root.right                     # cây phải thay chỗ root
```

**Lưu ý:** cách này **có thể làm cây cao hơn** (cây trái bị treo xuống dưới cùng) nhưng vẫn là BST hợp lệ và vẫn `O(h)` ở lần xoá.

In [12]:
def delete_graft(root, key):
    if root is None:
        return None
    if key < root.val:
        root.left = delete_graft(root.left, key)
        return root
    if key > root.val:
        root.right = delete_graft(root.right, key)
        return root
    if root.left is None:
        return root.right
    if root.right is None:
        return root.left
    node = root.right
    while node.left:
        node = node.left
    node.left = root.left               # treo cả cây trái
    return root.right


for vals, key in examples:
    out = check_delete(delete_graft, vals, key)
    print(vals, "xoá", key, "->", tree_to_list(out))

[5, 3, 6, 2, 4, None, 7] xoá 3 -> [5, 4, 6, 2, None, None, 7]
[5, 3, 6, 2, 4, None, 7] xoá 0 -> [5, 3, 6, 2, 4, None, 7]
[] xoá 0 -> []


**Độ phức tạp.** Thời gian `O(h)` (tìm nút + đi xuống nút trái nhất), bộ nhớ `O(h)` đệ quy. Nhược điểm: chiều cao có thể tăng.

So sánh chiều cao sau khi xoá gốc (một cây có hai nhánh đều cao):

In [13]:
def height(root):
    h, level = 0, [root] if root else []
    while level:
        h += 1
        level = [c for n in level for c in (n.left, n.right) if c]
    return h


vals = [8, 4, 12, 2, 6, 10, 14, 1, 3, 5, 7, 9, 11, 13, 15]       # cây đầy đủ cao 4
print("cao ban đầu       :", height(build_tree(vals)))
print("copy kế nhiệm     :", height(delete_copy_successor(build_tree(vals), 8)))
print("treo cây trái     :", height(delete_graft(build_tree(vals), 8)))

cao ban đầu       : 4
copy kế nhiệm     : 4
treo cây trái     : 6


### Cách 4 — Thay bằng **chính nút tiền nhiệm** (nối con trỏ, không sao chép giá trị)

**Ý tưởng.** Nếu các nút là đối tượng lớn hoặc nơi khác giữ tham chiếu tới nút, **sao chép giá trị** (Cách 2) là không ổn. Thay vào đó ta **chuyển nguyên nút tiền nhiệm** `p` (phải nhất của cây trái) lên chỗ nút bị xoá:

1. Tìm `p` và cha của nó `par`.
2. Nếu `p` **không phải** con trực tiếp của nút xoá: gỡ `p` khỏi chỗ cũ bằng `par.right = p.left` (`p` không có con phải nên chỉ cần nhấc `p.left` lên), rồi `p.left = root.left`.
3. `p.right = root.right`; trả `p` thay cho `root`.

Đây là đối xứng gương của "dùng kế nhiệm" nhưng khác ở chỗ **không đổi giá trị của bất kỳ nút nào**.

In [14]:
def delete_relink_pred(root, key):
    if root is None:
        return None
    if key < root.val:
        root.left = delete_relink_pred(root.left, key)
        return root
    if key > root.val:
        root.right = delete_relink_pred(root.right, key)
        return root
    if root.left is None:
        return root.right
    if root.right is None:
        return root.left
    par, pred = root, root.left
    while pred.right:                      # tiền nhiệm = phải nhất của cây trái
        par, pred = pred, pred.right
    if par is not root:
        par.right = pred.left              # gỡ pred khỏi chỗ cũ, nhấc con trái của nó lên
        pred.left = root.left
    pred.right = root.right
    return pred


for vals, key in examples:
    out = check_delete(delete_relink_pred, vals, key)
    print(vals, "xoá", key, "->", tree_to_list(out))

# giá trị của các nút KHÁC không bị đổi: kiểm tra định danh nút
t = build_tree([5, 3, 6, 2, 4, None, 7])
node2 = t.left.left
out = delete_relink_pred(t, 3)
assert out.left is node2 and node2.val == 2        # nút "2" vẫn là chính đối tượng cũ
print("sau khi xoá 3 (tiền nhiệm 2 lên thay):", tree_to_list(out))

[5, 3, 6, 2, 4, None, 7] xoá 3 -> [5, 2, 6, None, 4, None, 7]
[5, 3, 6, 2, 4, None, 7] xoá 0 -> [5, 3, 6, 2, 4, None, 7]
[] xoá 0 -> []
sau khi xoá 3 (tiền nhiệm 2 lên thay): [5, 2, 6, None, 4, None, 7]


**Độ phức tạp.** Thời gian `O(h)`, bộ nhớ `O(h)` đệ quy. Không sao chép giá trị → an toàn khi nơi khác còn tham chiếu nút.

### Cách 5 — **Vòng lặp** với con trỏ cha (`O(1)` bộ nhớ phụ)

**Ý tưởng.** Bỏ đệ quy: đi từ gốc xuống bằng `while`, nhớ **cha** `parent` của nút hiện tại. Khi tìm thấy `cur`, tính **nút thay thế** `repl` theo ba trường hợp (như Cách 3 cho hai con) rồi **cập nhật con của `parent`** (trái hay phải?) hoặc đổi gốc nếu `parent` là `None`. Ưu điểm: không đệ quy → không lo `RecursionError` với cây lệch 10^4 nút; bộ nhớ phụ `O(1)`.

**Giải thích từng đoạn.**

1. Đi xuống: `while cur and cur.val != key: parent, cur = cur, (cur.left if key < cur.val else cur.right)`.
2. `cur is None` → không có key, trả `root`.
3. Tính `repl`: không con trái → `cur.right`; không con phải → `cur.left`; hai con → treo `cur.left` vào trái nhất của `cur.right`, `repl = cur.right`.
4. Nối `repl` vào `parent` đúng phía (`parent.left is cur` hay không); nếu `parent is None` thì `repl` là gốc mới.

In [15]:
def delete_iterative(root, key):
    parent, cur = None, root
    while cur and cur.val != key:
        parent = cur
        cur = cur.left if key < cur.val else cur.right
    if cur is None:                      # không có key
        return root
    if cur.left is None:
        repl = cur.right
    elif cur.right is None:
        repl = cur.left
    else:
        node = cur.right
        while node.left:
            node = node.left
        node.left = cur.left             # treo cây trái vào nút trái nhất của cây phải
        repl = cur.right
    if parent is None:                   # xoá gốc: repl là gốc mới
        return repl
    if parent.left is cur:
        parent.left = repl
    else:
        parent.right = repl
    return root


for vals, key in examples:
    out = check_delete(delete_iterative, vals, key)
    print(vals, "xoá", key, "->", tree_to_list(out))

[5, 3, 6, 2, 4, None, 7] xoá 3 -> [5, 4, 6, 2, None, None, 7]
[5, 3, 6, 2, 4, None, 7] xoá 0 -> [5, 3, 6, 2, 4, None, 7]
[] xoá 0 -> []


**Độ phức tạp.** Thời gian `O(h)`, bộ nhớ `O(1)`. Đạt đúng yêu cầu follow-up và chống tràn đệ quy.

## 8. TRACE từng bước

Ta trace **Cách 3** (đệ quy, treo cây trái) rồi **Cách 5** (vòng lặp) khi xoá `3` khỏi `[5,3,6,2,4,null,7]`.

In [16]:
def trace_graft(root, key, depth=0):
    pad = "  " * depth
    if root is None:
        print(f"{pad}gặp None -> không có key, trả None")
        return None
    print(f"{pad}ở nút {root.val}, key={key}", end=": ")
    if key < root.val:
        print(f"{key} < {root.val} -> đi TRÁI")
        root.left = trace_graft(root.left, key, depth + 1)
        print(f"{pad}quay về {root.val}: root.left = {root.left.val if root.left else None}")
        return root
    if key > root.val:
        print(f"{key} > {root.val} -> đi PHẢI")
        root.right = trace_graft(root.right, key, depth + 1)
        print(f"{pad}quay về {root.val}: root.right = {root.right.val if root.right else None}")
        return root
    print("TÌM THẤY")
    if root.left is None:
        print(f"{pad}  không có con trái -> trả con phải {root.right.val if root.right else None}")
        return root.right
    if root.right is None:
        print(f"{pad}  không có con phải -> trả con trái {root.left.val}")
        return root.left
    node = root.right
    while node.left:
        node = node.left
    print(f"{pad}  hai con: trái nhất của cây phải = {node.val}; treo cây trái (gốc {root.left.val}) vào {node.val}.left")
    node.left = root.left
    print(f"{pad}  trả cây phải (gốc {root.right.val}) thay cho {root.val}")
    return root.right


t = build_tree([5, 3, 6, 2, 4, None, 7])
t = trace_graft(t, 3)
print("kết quả:", tree_to_list(t), "hợp lệ:", is_valid_bst(t))
assert is_valid_bst(t) and inorder(t) == [2, 4, 5, 6, 7]

ở nút 5, key=3: 3 < 5 -> đi TRÁI
  ở nút 3, key=3: TÌM THẤY
    hai con: trái nhất của cây phải = 4; treo cây trái (gốc 2) vào 4.left
    trả cây phải (gốc 4) thay cho 3
quay về 5: root.left = 4
kết quả: [5, 4, 6, 2, None, None, 7] hợp lệ: True


In [17]:
def trace_iterative(root, key):
    parent, cur = None, root
    step = 0
    while cur and cur.val != key:
        step += 1
        d = "TRÁI" if key < cur.val else "PHẢI"
        print(f"bước {step}: cur={cur.val}, parent={parent.val if parent else None} -> đi {d}")
        parent = cur
        cur = cur.left if key < cur.val else cur.right
    if cur is None:
        print("không có key"); return root
    print(f"tìm thấy cur={cur.val}, parent={parent.val if parent else None}, "
          f"left={cur.left.val if cur.left else None}, right={cur.right.val if cur.right else None}")
    if cur.left is None:
        repl = cur.right
    elif cur.right is None:
        repl = cur.left
    else:
        node = cur.right
        while node.left:
            node = node.left
        print(f"hai con: nút trái nhất của cây phải = {node.val}; node.left = cur.left ({cur.left.val})")
        node.left = cur.left
        repl = cur.right
    print(f"nút thay thế repl = {repl.val if repl else None}")
    if parent is None:
        return repl
    side = "left" if parent.left is cur else "right"
    setattr(parent, side, repl)
    print(f"nối: parent({parent.val}).{side} = {repl.val if repl else None}")
    return root


t = build_tree([5, 3, 6, 2, 4, None, 7])
t = trace_iterative(t, 3)
print("kết quả:", tree_to_list(t))
assert is_valid_bst(t) and inorder(t) == [2, 4, 5, 6, 7]

bước 1: cur=5, parent=None -> đi TRÁI
tìm thấy cur=3, parent=5, left=2, right=4
hai con: nút trái nhất của cây phải = 4; node.left = cur.left (2)
nút thay thế repl = 4
nối: parent(5).left = 4
kết quả: [5, 4, 6, 2, None, None, 7]


Bảng trace Cách 5 (xoá `3`):

| Bước | `parent` | `cur` | Quyết định |
|---|---|---|---|
| 1 | `None` | 5 | `3 < 5` → trái |
| (dừng) | 5 | 3 | `cur.val == key`, hai con (2 và 4) |
| tính `repl` | | | trái nhất của cây phải (`4`) là `4`; `4.left = 2`; `repl = 4` |
| nối | 5 | | `5.left is cur` → `5.left = 4` |

Cây kết quả: `[5,4,6,2,null,null,7]`; trung tự `2 4 5 6 7` (không còn `3`). Trường hợp xoá **gốc** `5`: `parent is None`, `repl = 6` trở thành gốc mới sau khi treo cây trái `3...` vào trái nhất của `6` (chính là `6`) — thử ở bài tập.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 5 (vòng lặp)**: `O(h)` thời gian, `O(1)` bộ nhớ phụ, không lo tràn đệ quy với cây lệch. Nếu muốn ngắn gọn dễ đọc, bản đệ quy Cách 2/Cách 3 ở mục 7 cũng được chấp nhận.

In [18]:
class Solution:
    def deleteNode(self, root: Optional[TreeNode], key: int) -> Optional[TreeNode]:
        parent, cur = None, root
        while cur and cur.val != key:
            parent = cur
            cur = cur.left if key < cur.val else cur.right
        if cur is None:
            return root
        if cur.left is None:
            repl = cur.right
        elif cur.right is None:
            repl = cur.left
        else:
            node = cur.right
            while node.left:
                node = node.left
            node.left = cur.left
            repl = cur.right
        if parent is None:
            return repl
        if parent.left is cur:
            parent.left = repl
        else:
            parent.right = repl
        return root


sol = Solution()
for vals, key in examples:
    out = check_delete(sol.deleteNode, vals, key)
    print(vals, "xoá", key, "->", tree_to_list(out))

[5, 3, 6, 2, 4, None, 7] xoá 3 -> [5, 4, 6, 2, None, None, 7]
[5, 3, 6, 2, 4, None, 7] xoá 0 -> [5, 3, 6, 2, 4, None, 7]
[] xoá 0 -> []


## 10. Vì sao thuật toán đúng

**Phần tìm kiếm.** Do tính chất BST, nếu `key` có trong cây thì nó nằm trên đường đi từ gốc: mỗi lần so sánh ta loại bỏ đúng cây con không thể chứa `key`.

**Phần xoá — ba trường hợp, mỗi trường hợp giữ trung tự tăng dần:**

- **Lá:** bỏ nút, dãy trung tự mất đúng phần tử `key`.
- **Một con:** nhấc con lên; dãy trung tự của cây con vẫn `L..., key` hoặc `key, R...` và chỉ mất `key`.
- **Hai con** (cây trái `L`, cây phải `R`, trung tự = `inorder(L), key, inorder(R)`):
  - *Cách 2/4:* thay bằng kế nhiệm/tiền nhiệm — là phần tử liền kề `key` trong trung tự, nên dãy mới là `inorder(L), inorder(R)` với phần tử liền kề được đưa lên làm gốc → vẫn tăng dần.
  - *Cách 3/5:* treo `L` vào trái nhất của `R`: nút trái nhất `m` của `R` là phần tử nhỏ nhất của `R`, nên `L` (toàn giá trị `< key < m`) đặt ở con trái của `m` đúng thứ tự: trung tự mới `inorder(L), m, ...` = `inorder(L), inorder(R)`.

**Tính chất bất biến (invariant):** sau khi xoá, trung tự = trung tự cũ bỏ `key`, và cây vẫn là cây (không chu trình, không nút dùng chung). Ta kiểm chứng đúng bất biến này ở mục 12 bằng `check_delete`.

Kiểm chứng nhanh bằng thí nghiệm: mọi cách cho cùng dãy trung tự dù hình dạng khác nhau.

In [19]:
vals = [5, 3, 6, 2, 4, None, 7]
methods = {
    "rebuild": delete_rebuild,
    "copy_successor": delete_copy_successor,
    "graft": delete_graft,
    "relink_pred": delete_relink_pred,
    "iterative": delete_iterative,
}
shapes = {}
for name, fn in methods.items():
    out = fn(build_tree(vals), 5)          # xoá GỐC (hai con)
    assert is_valid_bst(out) and inorder(out) == [2, 3, 4, 6, 7]
    shapes[name] = tree_to_list(out)
    print(f"{name:15s} {shapes[name]}")
print("số hình dạng khác nhau:", len({tuple(s) for s in shapes.values()}), "- đều hợp lệ")

rebuild         [4, 2, 6, None, 3, None, 7]
copy_successor  [6, 3, 7, 2, 4]
graft           [6, 3, 7, 2, 4]
relink_pred     [4, 3, 6, 2, None, None, 7]
iterative       [6, 3, 7, 2, 4]
số hình dạng khác nhau: 3 - đều hợp lệ


## 11. Phân tích Time / Space Complexity

Với `n` nút, `h` chiều cao (`log n ≤ h ≤ n`):

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Dựng lại từ trung tự | `O(n)` | `O(n)` | luôn duyệt mọi nút; phá hình dạng |
| 2. Copy kế nhiệm | `O(h)` | `O(h)` đệ quy | đổi giá trị nút |
| 3. Treo cây trái | `O(h)` | `O(h)` đệ quy | có thể tăng chiều cao |
| 4. Nối tiền nhiệm | `O(h)` | `O(h)` đệ quy | không copy giá trị |
| 5. Vòng lặp | `O(h)` | `O(1)` | không đệ quy |

Cây cân bằng: `h = O(log n)`; cây lệch: `h = O(n)`. Đạt follow-up `O(h)`.

## 12. Bộ test

### 12.1. Ví dụ của đề (kiểm tra bằng tính chất, không so đáp án cứng)

In [20]:
ALL = {
    "rebuild": (delete_rebuild, False),
    "copy_successor": (delete_copy_successor, True),
    "graft": (delete_graft, True),
    "relink_pred": (delete_relink_pred, True),
    "iterative": (delete_iterative, True),
    "Solution": (Solution().deleteNode, True),
}


def check_all(vals, key):
    for name, (fn, same) in ALL.items():
        check_delete(fn, vals, key, shape_if_absent=same)


for vals, key in examples:
    check_all(vals, key)
print("3 ví dụ của đề: tất cả các cách thoả bất biến.")

# Với ví dụ 1, đáp án hợp lệ nằm trong tập {2 đáp án nêu trong đề} cho các cách chuẩn
valid_answers = [[5, 4, 6, 2, None, None, 7], [5, 2, 6, None, 4, None, 7]]
for fn in (delete_copy_successor, delete_relink_pred, Solution().deleteNode):
    out = tree_to_list(fn(build_tree([5, 3, 6, 2, 4, None, 7]), 3))
    print(fn.__name__ if hasattr(fn, "__name__") else fn, "->", out)

3 ví dụ của đề: tất cả các cách thoả bất biến.
delete_copy_successor -> [5, 4, 6, 2, None, None, 7]
delete_relink_pred -> [5, 2, 6, None, 4, None, 7]
deleteNode -> [5, 4, 6, 2, None, None, 7]


### 12.2. Edge case

Cây rỗng, một nút (xoá chính nó / key khác), xoá gốc, xoá lá trái nhất/phải nhất, xoá nút có đúng một con, xoá nút hai con ở nhiều vị trí, các giá trị cực biên `±10^5`.

In [21]:
edge_cases = [
    ([], 0),
    ([1], 1),
    ([1], 2),
    ([2, 1], 2),                         # xoá gốc có đúng một con trái
    ([2, None, 3], 2),                   # xoá gốc có đúng một con phải
    ([2, 1, 3], 2),                      # xoá gốc có hai con
    ([2, 1, 3], 1),
    ([2, 1, 3], 3),
    ([5, 3, 6, 2, 4, None, 7], 5),
    ([5, 3, 6, 2, 4, None, 7], 2),
    ([5, 3, 6, 2, 4, None, 7], 6),
    ([5, 3, 6, 2, 4, None, 7], 7),
    ([5, 3, 6, 2, 4, None, 7], 4),
    ([5, 3, 6, 2, 4, None, 7], 100000),
    ([0, -100000, 100000], -100000),
    ([0, -100000, 100000], 100000),
    ([0, -100000, 100000], 0),
]
for vals, key in edge_cases:
    check_all(vals, key)
print(f"{len(edge_cases)} edge case: tất cả các cách đều đúng.")

17 edge case: tất cả các cách đều đúng.


### 12.3. Phản ví dụ cho các cách làm sai

Ta thử vài "cách làm sai" phổ biến và để `check_delete` **bắt** chúng:

- (a) Xoá nút hai con mà chỉ trả `root.left` → mất cả cây phải.
- (b) Thay bằng nút **lớn nhất của cây phải** (nhầm kế nhiệm) → phá BST.
- (c) Không gán lại `root.left = ...` → không xoá được nút sâu.

In [22]:
def wrong_lose_subtree(root, key):
    if root is None:
        return None
    if key < root.val:
        root.left = wrong_lose_subtree(root.left, key)
    elif key > root.val:
        root.right = wrong_lose_subtree(root.right, key)
    else:
        return root.left or root.right      # SAI: hai con -> mất một cây con
    return root


def wrong_max_of_right(root, key):
    if root is None:
        return None
    if key < root.val:
        root.left = wrong_max_of_right(root.left, key)
        return root
    if key > root.val:
        root.right = wrong_max_of_right(root.right, key)
        return root
    if root.left is None:
        return root.right
    if root.right is None:
        return root.left
    m = root.right
    while m.right:                          # SAI: lớn nhất cây phải, không phải nhỏ nhất
        m = m.right
    root.val = m.val
    root.right = wrong_max_of_right(root.right, m.val)
    return root


def wrong_no_assign(root, key):
    if root is None:
        return None
    if key < root.val:
        wrong_no_assign(root.left, key)     # SAI: không gán lại kết quả
        return root
    if key > root.val:
        wrong_no_assign(root.right, key)
        return root
    if root.left is None:
        return root.right
    if root.right is None:
        return root.left
    return root


for fn, vals, key in [
    (wrong_lose_subtree, [5, 3, 6, 2, 4, None, 7], 3),
    (wrong_max_of_right, [5, 3, 8, 2, 4, 6, 9, None, None, None, None, None, 7], 5),
    (wrong_no_assign, [5, 3, 6, 2, 4, None, 7], 2),
]:
    try:
        check_delete(fn, vals, key)
        print(fn.__name__, "- KHÔNG bị bắt (cần thêm test!)")
    except AssertionError as e:
        print(f"{fn.__name__:20s} bị bắt: {e.args[0][0]}")

wrong_lose_subtree   bị bắt: sai tập giá trị
wrong_max_of_right   bị bắt: không còn là BST
wrong_no_assign      bị bắt: sai tập giá trị


### 12.4. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

Oracle: tập giá trị mong đợi = `sorted(set(values) - {key})`; kèm kiểm tra BST hợp lệ. Cây ngẫu nhiên đúng ràng buộc (giá trị khác nhau trong `[-10^5, 10^5]`), `key` lấy 70% từ trong cây, 30% ngoài cây; thử cả key là gốc.

In [23]:
def random_bst_values(rng, n, lo=-10**5, hi=10**5):
    vals = rng.sample(range(lo, hi + 1), n)
    # dựng bằng chèn lần lượt, rồi xuất ra list kiểu LeetCode để dựng lại cây mới nhiều lần
    root = None
    for v in vals:
        if root is None:
            root = TreeNode(v)
            continue
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None:
                    cur.left = TreeNode(v); break
                cur = cur.left
            else:
                if cur.right is None:
                    cur.right = TreeNode(v); break
                cur = cur.right
    return tree_to_list(root), vals


rng = random.Random(450)
for trial in range(600):
    n = rng.randint(0, 40)
    tl, vals = random_bst_values(rng, n)
    if vals and rng.random() < 0.7:
        key = rng.choice(vals)
    else:
        key = rng.randint(-10**5, 10**5)
    check_all(tl, key)
    if tl:
        check_all(tl, tl[0])               # xoá gốc
print("600 cây ngẫu nhiên: mọi cách đúng bất biến (hợp lệ + đúng tập giá trị).")

600 cây ngẫu nhiên: mọi cách đúng bất biến (hợp lệ + đúng tập giá trị).


### 12.5. Xoá liên tiếp nhiều key trên cùng cây lớn

Với cây `2000` nút ngẫu nhiên, xoá lần lượt `500` key (ngẫu nhiên, có cả key đã xoá) và sau **mỗi** lần kiểm tra bất biến so với `set` thuần Python — mô phỏng một dãy thao tác thật.

In [24]:
for name, (fn, _) in ALL.items():
    rng = random.Random(4500)
    tl, vals = random_bst_values(rng, 2000)
    root = build_tree(tl)
    alive = set(vals)
    for step in range(500):
        key = rng.choice(vals) if rng.random() < 0.9 else rng.randint(-10**5, 10**5)
        root = fn(root, key)
        alive.discard(key)
        assert is_valid_bst(root) and inorder(root) == sorted(alive), (name, step, key)
    print(f"{name:15s}: 500 lần xoá liên tiếp OK, còn {len(alive)} nút, cao {height(root)}")

rebuild        : 500 lần xoá liên tiếp OK, còn 1594 nút, cao 11


copy_successor : 500 lần xoá liên tiếp OK, còn 1594 nút, cao 23


graft          : 500 lần xoá liên tiếp OK, còn 1594 nút, cao 32


relink_pred    : 500 lần xoá liên tiếp OK, còn 1594 nút, cao 23


iterative      : 500 lần xoá liên tiếp OK, còn 1594 nút, cao 32


Solution       : 500 lần xoá liên tiếp OK, còn 1594 nút, cao 32


### 12.6. Cây lệch 10^4 nút: đệ quy tràn, vòng lặp chạy tốt

Chuỗi phải 10^4 nút; xoá nút **ở đáy** (độ sâu 10^4). Các cách đệ quy với giới hạn mặc định `1000` sẽ `RecursionError`; cách vòng lặp thì không.

In [25]:
def chain_right(n):
    root = None
    for v in range(n, 0, -1):
        root = TreeNode(v, right=root)
    return root


N = 10**4
print("giới hạn đệ quy:", sys.getrecursionlimit())
for name in ["copy_successor", "graft", "relink_pred", "iterative", "Solution"]:
    fn = ALL[name][0]
    root = chain_right(N)
    t0 = time.perf_counter()
    try:
        out = fn(root, N)                       # xoá nút đáy
        assert is_valid_bst(out) and inorder(out) == list(range(1, N))
        print(f"{name:15s}: OK, {time.perf_counter() - t0:.4f}s")
    except RecursionError:
        print(f"{name:15s}: RecursionError (cây sâu {N})")

giới hạn đệ quy: 1000
copy_successor : RecursionError (cây sâu 10000)
graft          : RecursionError (cây sâu 10000)
relink_pred    : RecursionError (cây sâu 10000)
iterative      : OK, 0.0055s
Solution       : OK, 0.0049s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên gán lại kết quả đệ quy

In [26]:
def bad_no_assign(root, key):
    if root is None:
        return None
    if key < root.val:
        bad_no_assign(root.left, key)        # SAI: kết quả bị bỏ rơi
        return root
    if key > root.val:
        bad_no_assign(root.right, key)
        return root
    return root.right if root.left is None else root.left     # (bỏ qua hai con cho đơn giản)


t = build_tree([5, 3, 6, 2, 4, None, 7])
t = bad_no_assign(t, 2)                       # 2 là lá trái của 3
print("sau khi 'xoá' 2:", tree_to_list(t), "-> vẫn còn 2!")
assert 2 in inorder(t)

sau khi 'xoá' 2: [5, 3, 6, 2, 4, None, 7] -> vẫn còn 2!


**Vì sao sai:** hàm trả về **gốc mới** của cây con (có thể là nút khác hoặc `None`); nếu cha không nhận kết quả thì thay đổi bị mất. **Đúng:** `root.left = self.deleteNode(root.left, key)`.

### Lỗi 2: Xoá nút hai con mà chỉ trả một con

In [27]:
t = build_tree([5, 3, 6, 2, 4, None, 7])
out = wrong_lose_subtree(t, 3)
print("sai :", tree_to_list(out), "(mất nút 4)")
t = build_tree([5, 3, 6, 2, 4, None, 7])
out = delete_iterative(t, 3)
print("đúng:", tree_to_list(out))

sai : [5, 2, 6, None, None, None, 7] (mất nút 4)
đúng: [5, 4, 6, 2, None, None, 7]


**Vì sao sai:** nút `3` có **hai** con (`2`, `4`); `return root.left or root.right` chọn một và **bỏ rơi** cây con còn lại. **Đúng:** thay bằng kế nhiệm/tiền nhiệm hoặc treo cây trái vào nút trái nhất của cây phải.

### Lỗi 3: Thay bằng nút "lớn nhất của cây phải" (nhầm kế nhiệm)

Kế nhiệm là nút **nhỏ nhất** của cây phải (đi `left`), không phải lớn nhất.

In [28]:
t = build_tree([5, 3, 8, 2, 4, 6, 9, None, None, None, None, None, 7])
out = wrong_max_of_right(t, 5)
print("cây sai:", tree_to_list(out), "hợp lệ:", is_valid_bst(out))
assert not is_valid_bst(out)

t = build_tree([5, 3, 8, 2, 4, 6, 9, None, None, None, None, None, 7])
out = delete_copy_successor(t, 5)
print("cây đúng:", tree_to_list(out), "hợp lệ:", is_valid_bst(out))
assert is_valid_bst(out)

cây sai: [9, 3, 8, 2, 4, 6, None, None, None, None, None, None, 7] hợp lệ: False
cây đúng: [6, 3, 8, 2, 4, 7, 9] hợp lệ: True


**Vì sao sai:** gốc mới `9` lớn hơn các nút còn lại trong cây phải (`8`, `6`, `7`), vi phạm "cây phải của gốc phải lớn hơn gốc". **Đúng:** kế nhiệm là nút nhỏ nhất của cây phải (`6`).

### Lỗi 4: Sao chép kế nhiệm nhưng **quên xoá** nó ở chỗ cũ

In [29]:
def bad_forget_remove(root, key):
    if root is None:
        return None
    if key < root.val:
        root.left = bad_forget_remove(root.left, key); return root
    if key > root.val:
        root.right = bad_forget_remove(root.right, key); return root
    if root.left is None:
        return root.right
    if root.right is None:
        return root.left
    succ = root.right
    while succ.left:
        succ = succ.left
    root.val = succ.val                 # SAI: không xoá succ ở cây phải -> giá trị bị lặp
    return root


t = build_tree([5, 3, 6, 2, 4, None, 7])
out = bad_forget_remove(t, 5)
print("trung tự:", inorder(out), "(6 xuất hiện hai lần, 5 mất)")
assert inorder(out) != [2, 3, 4, 6, 7]

trung tự: [2, 3, 4, 6, 6, 7] (6 xuất hiện hai lần, 5 mất)


**Vì sao sai:** sau `root.val = succ.val` nút kế nhiệm vẫn còn trong cây phải nên giá trị bị **trùng**. **Đúng:** thêm `root.right = delete(root.right, succ.val)`.

### Lỗi 5: Không xử lý `root is None`

In [30]:
def bad_none(root, key):
    if key < root.val:                      # SAI: root có thể là None
        root.left = bad_none(root.left, key)
    return root


try:
    bad_none(build_tree([5]), 1)
except AttributeError as e:
    print("Lỗi:", e)
print(delete_iterative(None, 7))            # đúng: trả None

Lỗi: 'NoneType' object has no attribute 'val'
None


**Vì sao sai:** đến nhánh rỗng (key không tồn tại, hoặc cây rỗng ở Ví dụ 3), `None.val` gây lỗi. **Đúng:** kiểm tra `root is None` ngay đầu hàm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Sửa cây bằng đệ quy"** → hàm trả về **gốc cây con mới**, cha gán lại: `node.left = f(node.left)`.
- **Xoá trong BST**: nhớ **ba trường hợp theo số con**; chỉ hai con cần "người kế vị".
- **Nhiều đáp án đúng** → viết hàm kiểm tra tính chất (`is_valid_bst`, so tập giá trị) thay vì so list cứng.
- **Kế nhiệm = trái nhất của cây phải; tiền nhiệm = phải nhất của cây trái.** Nhớ "phải rồi trái hết" và "trái rồi phải hết".
- **Muốn `O(1)` bộ nhớ** → vòng lặp + con trỏ cha (Cách 5).
- **Treo cây trái vào trái nhất của cây phải** là mẹo ngắn nhất để viết (ba dòng), nhưng cây có thể cao hơn.
- Thử luôn: **xoá gốc**, **cây rỗng**, **key không có**, **chuỗi lệch**.

## 15. Biến thể của bài

1. **Chèn vào BST** (bài 701): đối xứng, đi xuống đến chỗ trống rồi gắn nút mới.
2. **Cắt tỉa BST theo khoảng `[low, high]`** (bài 669): xoá mọi nút ngoài khoảng.
3. **Xoá nhiều key / xoá cả khoảng** (xem bài tập 5).
4. **BST có giá trị trùng**: cần quy ước hướng cho giá trị trùng, và xoá một hay xoá tất cả?
5. **Xoá nút trong cây tự cân bằng (AVL/Red-Black)**: sau khi xoá phải cân bằng lại.
6. **Xoá nút trong cây nhị phân bất kỳ** (không phải BST): phải duyệt toàn cây; thường thay bằng nút sâu nhất–phải nhất.

Ví dụ nhỏ biến thể 2 (cắt tỉa theo khoảng):

In [31]:
def trim_bst(root, low, high):
    if root is None:
        return None
    if root.val < low:                       # cả nút và cây trái đều quá nhỏ
        return trim_bst(root.right, low, high)
    if root.val > high:
        return trim_bst(root.left, low, high)
    root.left = trim_bst(root.left, low, high)
    root.right = trim_bst(root.right, low, high)
    return root


t = trim_bst(build_tree([5, 3, 6, 2, 4, None, 7]), 3, 6)
print(tree_to_list(t), inorder(t))
assert inorder(t) == [3, 4, 5, 6] and is_valid_bst(t)

[5, 3, 6, None, 4] [3, 4, 5, 6]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng lại | trung tự − `key` → dựng cân bằng | `O(n)` | `O(n)` | oracle kiểm tra; dữ liệu nhỏ |
| 2. Copy kế nhiệm | chép giá trị kế nhiệm, xoá kế nhiệm | `O(h)` | `O(h)` | giải thích kinh điển, dễ chứng minh |
| 3. Treo cây trái | `node.left = root.left` ở trái nhất cây phải | `O(h)` | `O(h)` | code ngắn nhất; chấp nhận cây cao hơn |
| 4. Nối tiền nhiệm | chuyển nguyên nút tiền nhiệm lên | `O(h)` | `O(h)` | khi không được đổi giá trị nút |
| 5. Vòng lặp | `parent` + `repl` | `O(h)` | `O(1)` | cây sâu, tránh đệ quy, nộp bài |

Quy tắc chọn: **thi đấu / phỏng vấn** → Cách 2 hoặc 3 (ngắn, dễ giải thích); **cây rất sâu** → Cách 5.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `tree_to_list(delete_iterative(build_tree([2, 1, 3]), 1))`, `tree_to_list(delete_iterative(build_tree([2, 1, 3]), 5))`, `tree_to_list(delete_iterative(build_tree([1]), 1))`, `inorder(delete_graft(build_tree([4, 2, 6, 1, 3, 5, 7]), 4))`, `leftmost(build_tree([4, 2, 6, 1, 3, 5, 7]).right).val`.

**Bài 2 (tính tay).** Cây `[8,3,10,1,6,null,14,null,null,4,7,13]`. (a) Phân loại số con của các nút `1, 14, 3, 8`. (b) Vẽ hai cây kết quả khi xoá `3` (dùng kế nhiệm và dùng tiền nhiệm), viết dạng list. (c) Xoá `8` bằng cách "treo cây trái vào trái nhất của cây phải": cây cao bao nhiêu?

**Bài 3 (điền TODO).** Hoàn thành `delete_todo(root, key)` theo Cách 2 (copy kế nhiệm).

In [32]:
def delete_todo(root, key):
    # TODO 1: nếu root là None -> trả None
    # TODO 2: nếu key < root.val -> root.left = delete_todo(root.left, key); trả root
    #         nếu key > root.val -> tương tự với root.right
    # TODO 3: root.val == key: nếu thiếu con trái -> trả root.right; thiếu con phải -> trả root.left
    # TODO 4: hai con: tìm succ = nút trái nhất của root.right; root.val = succ.val;
    #         root.right = delete_todo(root.right, succ.val); trả root
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `delete_min(root)` trả về cặp `(gốc_mới, giá_trị_nhỏ_nhất_đã_xoá)` bằng **vòng lặp** (không đệ quy). Sau đó viết lại nhánh hai con của Cách 2 dùng `delete_min(root.right)` thay vì gọi đệ quy xoá `succ.val`.

**Bài 5 (biến thể / mini challenge).** Viết `delete_range(root, lo, hi)` xoá **mọi** nút có giá trị trong `[lo, hi]` và trả về gốc mới. Kiểm tra: kết quả là BST hợp lệ và `inorder` = các giá trị ngoài `[lo, hi]`. Gợi ý: kết hợp trim (mục 15) nhưng ngược lại, và xử lý nút nằm trong khoảng bằng cách "ghép" hai cây con.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[2, None, 3]`, `[2, 1, 3]`, `[]`, `[1, 2, 3, 5, 6, 7]` (trung tự của cây gốc `4` bỏ `4`), và `5` (trái nhất của cây phải gốc `6` là `5`). Bài 3: bám sát mã Cách 2 mục 7.
- **Mức 2 (vừa):** Bài 2: (a) `1` lá; `14` một con (trái `13`); `3` hai con; `8` hai con. (b) kế nhiệm của `3` là `4` (trái nhất của cây phải gốc `6`); tiền nhiệm là `1`. (c) trái nhất của cây phải (gốc `10`) là `10`; treo cây trái gốc `3` vào `10.left`: chiều cao thay đổi, hãy tự đếm bằng `height`. Bài 4: dùng `parent` khi đi xuống trái nhất; nếu nút gốc chính là trái nhất thì gốc mới là `root.right`.
- **Mức 3 (sâu):** Bài 5: hàm `f(node)`: nếu `node.val < lo` → `node.right = f(node.right)`, trả `node`; nếu `node.val > hi` → `node.left = f(node.left)`, trả `node`; nếu trong `[lo, hi]` → xoá cả nút: `L = f(node.left)`, `R = f(node.right)` rồi **ghép** `L` và `R` (mọi giá trị của `L` nhỏ hơn mọi giá trị của `R`): dùng cách treo `L` vào trái nhất của `R`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert tree_to_list(delete_iterative(build_tree([2, 1, 3]), 1)) == [2, None, 3]
assert tree_to_list(delete_iterative(build_tree([2, 1, 3]), 5)) == [2, 1, 3]
assert tree_to_list(delete_iterative(build_tree([1]), 1)) == []
assert inorder(delete_graft(build_tree([4, 2, 6, 1, 3, 5, 7]), 4)) == [1, 2, 3, 5, 6, 7]
assert leftmost(build_tree([4, 2, 6, 1, 3, 5, 7]).right).val == 5

# Bài 2
t2 = build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13])
assert [classify(find(t2, k)) for k in (1, 14, 3, 8)] == ["lá (0 con)", "một con", "hai con", "hai con"]
a = delete_copy_successor(build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]), 3)
b = delete_relink_pred(build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]), 3)
print("xoá 3, kế nhiệm :", tree_to_list(a))
print("xoá 3, tiền nhiệm:", tree_to_list(b))
assert tree_to_list(a) == [8, 4, 10, 1, 6, None, 14, None, None, None, 7, 13]
assert tree_to_list(b) == [8, 1, 10, None, 6, None, 14, 4, 7, 13]
c = delete_graft(build_tree([8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]), 8)
print("xoá 8 bằng treo cây trái: cao", height(c), tree_to_list(c))
assert is_valid_bst(c) and inorder(c) == [1, 3, 4, 6, 7, 10, 13, 14]


# Bài 3
def delete_todo_solution(root, key):
    if root is None:
        return None
    if key < root.val:
        root.left = delete_todo_solution(root.left, key)
        return root
    if key > root.val:
        root.right = delete_todo_solution(root.right, key)
        return root
    if root.left is None:
        return root.right
    if root.right is None:
        return root.left
    succ = root.right
    while succ.left:
        succ = succ.left
    root.val = succ.val
    root.right = delete_todo_solution(root.right, succ.val)
    return root


for vals, key in edge_cases:
    check_delete(delete_todo_solution, vals, key)


# Bài 4
def delete_min(root):
    """Trả (gốc mới, giá trị nhỏ nhất đã xoá). Giả sử root không rỗng."""
    if root.left is None:
        return root.right, root.val
    parent, cur = root, root.left
    while cur.left:
        parent, cur = cur, cur.left
    parent.left = cur.right
    return root, cur.val


def delete_with_delete_min(root, key):
    parent, cur = None, root
    while cur and cur.val != key:
        parent, cur = cur, (cur.left if key < cur.val else cur.right)
    if cur is None:
        return root
    if cur.left is None:
        repl = cur.right
    elif cur.right is None:
        repl = cur.left
    else:
        cur.right, cur.val = delete_min(cur.right)        # kế nhiệm: lấy giá trị, xoá khỏi cây phải
        return root
    if parent is None:
        return repl
    if parent.left is cur:
        parent.left = repl
    else:
        parent.right = repl
    return root


for vals, key in edge_cases:
    check_delete(delete_with_delete_min, vals, key)
r, m = delete_min(build_tree([5, 3, 6, 2, 4, None, 7]))
assert m == 2 and inorder(r) == [3, 4, 5, 6, 7]


# Bài 5
def delete_range(root, lo, hi):
    def merge(L, R):
        if L is None:
            return R
        if R is None:
            return L
        node = R
        while node.left:
            node = node.left
        node.left = L
        return R

    def f(node):
        if node is None:
            return None
        if node.val < lo:
            node.right = f(node.right)
            return node
        if node.val > hi:
            node.left = f(node.left)
            return node
        return merge(f(node.left), f(node.right))

    return f(root)


rng = random.Random(1717)
for _ in range(300):
    tl, vals = random_bst_values(rng, rng.randint(0, 30), -50, 50)
    lo = rng.randint(-50, 50)
    hi = lo + rng.randint(0, 40)
    out = delete_range(build_tree(tl), lo, hi)
    assert is_valid_bst(out)
    assert inorder(out) == sorted(v for v in vals if not (lo <= v <= hi))
print("Tất cả đáp án bài tập đều đúng.")

xoá 3, kế nhiệm : [8, 4, 10, 1, 6, None, 14, None, None, None, 7, 13]
xoá 3, tiền nhiệm: [8, 1, 10, None, 6, None, 14, 4, 7, 13]
xoá 8 bằng treo cây trái: cao 4 [10, 3, 14, 1, 6, 13, None, None, None, 4, 7]
Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xoá trong BST, đệ quy "trả về gốc cây con", treo cây trái)

```python
class Solution:
    def deleteNode(self, root: Optional[TreeNode], key: int) -> Optional[TreeNode]:
        if root is None:
            return None
        if key < root.val:
            root.left = self.deleteNode(root.left, key)
            return root
        if key > root.val:
            root.right = self.deleteNode(root.right, key)
            return root
        if root.left is None:            # lá hoặc chỉ có con phải
            return root.right
        if root.right is None:           # chỉ có con trái
            return root.left
        node = root.right                # hai con
        while node.left:
            node = node.left
        node.left = root.left            # treo cả cây trái vào trái nhất của cây phải
        return root.right
```

**Checklist**

- [ ] Có xử lý `root is None` (cây rỗng / không tìm thấy)?
- [ ] Gán lại kết quả đệ quy: `root.left = ...`, `root.right = ...`?
- [ ] Ba trường hợp theo số con: lá, một con, hai con — đã đủ?
- [ ] Nút hai con: kế nhiệm = **trái nhất của cây phải** (hoặc tiền nhiệm = phải nhất của cây trái)?
- [ ] Nếu copy giá trị kế nhiệm thì đã **xoá kế nhiệm ở chỗ cũ**?
- [ ] Đã thử xoá **gốc**, key không tồn tại, cây một nút, chuỗi lệch?
- [ ] Test bằng tính chất (BST hợp lệ + đúng tập giá trị) vì có nhiều đáp án đúng?
- [ ] Skeleton đúng `class Solution` / `def deleteNode(self, root: Optional[TreeNode], key: int) -> Optional[TreeNode]`?